<a href="https://colab.research.google.com/github/ChrisZweck/slurm_utilities/blob/main/Tsunami_HySEA_MessinaWorkshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center>

<table style="width:100%; border:0px; border-collapse:collapse; table-layout: fixed;">
  <tr>
    <td colspan="2" style="text-align:center; border:0px; padding-bottom: 20px;">
      <img src="https://www.geo-inquire.eu/fileadmin/geoinquire/branding/Geo-INQUIRE_logo.jpg" width="500px">
    </td>
  </tr>
  <tr>
    <td style="width:50%; text-align:left; border:0px; vertical-align: middle;">
      <img src="https://www.uma.es/media/fotos/image_30021.jpeg" width="350px">
    </td>
    <td style="width:50%; text-align:right; border:0px; vertical-align: middle;">
      <img src="https://www.uma.es/media/tinyimages/img/image_12760.png" width="150px">
    </td>
  </tr>
</table>
<hr>
</center>

# **Training Session: Earthquake-Generated Tsunami Simulations with Tsunami-HySEA**

Trainers: Alejandro González (UMA, alexgp@uma.es), José Manuel González (UMA, jgv@uma.es)

# **Introduction**

In this **training course**, we will learn the basics of using the **Tsunami-HySEA** simulation software to perform earthquake-triggered tsunami simulations.

### **Course Goals:**
* **Data Acquisition:** Learn how to download public elevation data.
* **Preprocessing:** Prepare and adapt data to the specific formats required by Tsunami-HySEA.
* **Configuration:** Understand and generate the parameter file to run a tsunami propagation simulation.
* **HPC Execution:** Launch the simulation in a **High-Performance Computing (HPC)** environment.
* **Post-processing:** Analyze and visualize simulation results for clear interpretation.
* **Inundation simulation:** Prepare data (topobathymetries + parameter files) for coastal flooding simulations.

---

# **1. Numerical Framework: What is Tsunami-HySEA?**

**Tsunami-HySEA** is a member of the HySEA family designed for the accurate and efficient simulation of tsunamis triggered by earthquakes.

### **Core elements**
Tsunami-HySEA implements numerical schemes based on the **Finite Volume Method** to approximate the solution of the **Shallow Water Equations (SWE)**.
* Single-layer hydrostatic fluids.
* GPU-accelerated code, implemented in CUDA language.
* The software is able to reproduce the lifecycle of a seismic tsunami event: **Initial condition**, **Propagation** and **Inundation**.


---


# **2. Prerequisites to Run a Simulation Using Tsunami-HySEA**

Tsunami-HySEA requires at least two main components:

1.  **Topobathymetry file** on a rectangular grid.
2.  **Parameter file** (configuration file).



### **Topobathymetry File Requirements**
The topobathymetry must be a NetCDF file (`.nc`, `.grd`) in **netcdf4** format, containing the following variables:

| Standard Variable | Alternative Format |
| :--- | :--- |
| `float64 lon(lon)` | `float64 x(x)` |
| `float64 lat(lat)` | `float64 y(y)` |
| `float32 z(lat, lon)` | `float32 z(y, x)` |



Tsunami-HySEA is designed to propagate tsunami waves over long distances. Therefore, it is implemented in **spherical coordinates**, accounting for how the Earth's curvature affects the wave. For this reason, it uses geographic coordinates (Longitude, Latitude), specifically the **WGS 1984 (EPSG:4326)** reference system.

Fortunately, most major free topobathymetric data providers (GEBCO, EMODnet, etc.) utilize this coordinate system.



### **The Parameter File**
The parameter file controls the simulation details. It contains key information divided into several blocks:

* **Earthquake Rupture:** Fault information and **Okada parameters**.
* **Grids:** Topobathymetry file paths.
* **Time Control:** Total simulation time and data saving frequency.
* **Virtual Buoys (POIs):** Locations of points of interest to extract time series.
* **Numerical Parameters:** Stability condition ($CFL$), flooding thresholds, numerical scheme switch (WAF) and adimensionalization values.

> **Important:** The order of the values in the parameter file is **fundamental and fixed**.

### **Example Parameter File:**

```text
MEDITERRANEAN             # name of the experiment
EM_450m_topobathy.grd     # topobathymetry file
1                         # initialization mode 1: Okada Standard
0                         # apply kajura filter?
1                         # number of faults
0.0 14.9887 37.1526 9.802863566673047 85.84465449734515 22.98264453386507 67.5 50.0 90.0 3.9955153462891992                         # okada parameters (time,lon,lat,depth,length,width,strike,dip,rake,slip)
0                         # crop deformation?
simulation_EM_propagation # output filename
1 1 0 0 0 0 0 0 0 1       # output variables (eta,maxEta,vel,maxVel,maxModVel,maxModMassFlow,momFlux,maxMomFlux,AT)
1				                  #number of levels
1                         # upper boundary condition (1:open, -1:wall)
1                         # lower boundary condition (1:open, -1:wall)
1                         # left boundary condition  (1:open, -1:wall)
1                         # right boundary condition (1:open, -1:wall)
14400.1                   # total simulation time (in seconds)
600.0                     # netcdf saving time (in seconds)
1                         # read points from file? (for timeseries)
POIs.dat                  # file with the location of Points Of Interest
60.0                      # timeseries saving time
0.5                       # CFL
5e-3                      # epsilon_h (meters)
20.0                      # threshold for WAF scheme (meters)
0.2                       # stability coefficient
0                         # friction type
0.02                      # Manning bottom friction
100.0                     # maximum allowed velocity of water (m/s)
1000000.0                 # typical length (meters)
1000.0                    # typical height (meters)
1e-3                      # threshold for arrival times (meters)
```


TIP: **To know how to format your parameters file, run in the command line ./TsunamiHySEA**

---

# **3. Data Acquisition**

### **Resolution Requirements: What resolution I need?**
Earthquake-triggered tsunami waves have wavelengths of the order of ~100km. Then, for **propagation simulations**, having a grid with spatial resolution of several kilometers is more than enough to resolve the wave form.

For the example in this session, we will use data from **GEBCO**, which provides topobathymetric data for the entire Earth at **15 arc-seconds** resolution (approximately 450m).
* **URL:** [GEBCO Gridded Bathymetry Data](https://www.gebco.net/data-products/gridded-bathymetry-data)



### **Target Dataset: Sub-ice Topo/Bathy**
Specifically, we are interested in the **Sub-ice** model. This model "removes" the ice to show the true bedrock elevation and the actual seafloor beneath glaciers.



For **Tsunami-HySEA**, this is vital because it defines the **actual water column depth ($d$)** used in the **Shallow Water Equations (SWE)**.

### **Raster file covering the area of interest**
Upon unzipping the GEBCO download, you will find 8 raster files in `.tif` format; each one is a "tile" covering a specific part of the Earth. We will use the tile covering the **Eastern Mediterranean Sea**:

* `'gebco_2025_sub_ice_n90.0_s0.0_w0.0_e90.0.tif'`

---

## **3.1 Downloading Data for the current Session**

In [1]:
#This Python code snippet downloads the neccesary data for this session

import os
import io
import shutil
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from tqdm.notebook import tqdm

FOLDER_ID = '1KdX8syBSzBUiza4j-IxvTzc6dLeRQOy2'
LOCAL_DIR = 'MessinaWorkshop_THySEA'  #name of the local folder to be created

os.makedirs(LOCAL_DIR, exist_ok=True)

print("1. Authenticating user...")
auth.authenticate_user()
service = build('drive', 'v3')

print(f"2. Querying files in Folder ID: {FOLDER_ID}...")

results = service.files().list(
    q=f"'{FOLDER_ID}' in parents and trashed=false and mimeType != 'application/vnd.google-apps.folder'",
    fields="files(id, name, size)",
    supportsAllDrives=True,
    includeItemsFromAllDrives=True
).execute()

items = results.get('files', [])

for item in items:
    file_id = item['id']
    file_name = item['name']
    # Handle cases where size might be unknown (0)
    file_size = int(item.get('size', 0))

    filepath = os.path.join(LOCAL_DIR, file_name)

    print(f"\n⬇️ Downloading: {file_name} ({file_size/1024/1024:.2f} MB)")

    request = service.files().get_media(fileId=file_id)
    fh = io.FileIO(filepath, 'wb')
    downloader = MediaIoBaseDownload(fh, request)

    done = False
    with tqdm(total=100, unit="%", leave=False) as pbar:
        while not done:
            status, done = downloader.next_chunk()
            if status:
                pbar.update(int(status.progress() * 100) - pbar.n)

    print(f"   Shapefile/Data saved: {file_name}")


if os.path.exists("client_secrets.json"):
    os.remove("client_secrets.json")
    print("\n🧹 Cleanup: 'client_secrets.json' removed.")

print(f"\n🎉 Process Complete! Files are ready in: {LOCAL_DIR}")

1. Authenticating user...


MessageError: Error: credential propagation was unsuccessful

---


## **3.2 Manual Data Download**

If you were unable to download the data directly using the provided scripts, please use the following link to access the necessary files:

* **Download Link:** [Google Drive - Messina Workshop Data](https://drive.google.com/drive/folders/1KdX8syBSzBUiza4j-IxvTzc6dLeRQOy2?dmr=1&ec=wgc-drive-%5Bmodule%5D-goto)

Then upload the data manually to the created folder `MessinaWorkshop_THySEA`

---

# **Quick visualization of the downloaded raster**

The following Python snippet uses rasterio and matplotlib to load and visualize the GEBCO GeoTIFF file. It includes a **spatial downsampling** (coarsening) to ensure the map renders efficiently in a notebook environment.

In [ ]:
import rasterio
from rasterio.enums import Resampling
import matplotlib.pyplot as plt
import numpy as np
import os

file_path = '/content/MessinaWorkshop_THySEA/gebco_2025_sub_ice_n90.0_s0.0_w0.0_e90.0.tif'

with rasterio.open(file_path) as src:
    # Define reduction factor (coarsen by 10)
    reduction_factor = 10

    # Calculate the new dimensions for the downsampled grid
    new_height = src.height // reduction_factor
    new_width = src.width // reduction_factor

    # Corrected read call using rasterio.enums.Resampling
    da_lowres = src.read(
        1,
        out_shape=(new_height, new_width),
        resampling=Resampling.average
    ).astype(float)

    # Handle NoData values manually to avoid scale distortion
    if src.nodata is not None:
        da_lowres[da_lowres == src.nodata] = np.nan

    # Get spatial bounds for the plot axes
    bounds = src.bounds

    # --- Plotting Configuration ---
    plt.figure(figsize=(15, 10))

    img = plt.imshow(
        da_lowres,
        cmap='terrain',
        extent=[bounds.left, bounds.right, bounds.bottom, bounds.top]
    )

    # --- Formatting the plot ---
    cbar = plt.colorbar(img)
    cbar.set_label('Elevation / Depth (meters)')

    plt.title("GEBCO 2025 Global Bathymetry - Downsampled (Rasterio)", fontsize=15)
    plt.xlabel("Longitude [degrees_east]")
    plt.ylabel("Latitude [degrees_north]")
    plt.grid(True, linestyle=':', alpha=0.6)

    plt.show()

    # --- Dataset Metadata Output ---
    print("-" * 40)
    print(f"Original Full Resolution: ({src.height}, {src.width})")
    print(f"Plotting Resolution: {da_lowres.shape}")
    print(f"CRS: {src.crs}")
    print(f"Bounds: {bounds}")
    print("-" * 40)

---

# **4. Reformatting the Topobathymetry File**

To adapt the downloaded data to the specific format required by **Tsunami-HySEA**, we use the following script. This process performs two critical tasks:

1.  **Spatial Clipping:** It crops the large GEBCO `.tif` file to the specific geographic area defined by our coordinates (Bounding Box).
2.  **Format Conversion:** It converts the resulting raster from GeoTIFF to **.grd (NetCDF4)** format.

### **Why is this step necessary?**
* **Compatibility:** Tsunami-HySEA specifically looks for NetCDF structures where dimensions and variables are clearly defined as `x`, (or `lon`), `y`, (or `lat`), and `z`.

In [ ]:
print("Installing netCDF4... \n")
!pip install -q netCDF4
print("\n🎉 Installation Complete!")

import rasterio
from rasterio.windows import from_bounds
from netCDF4 import Dataset
import numpy as np
import os
from datetime import datetime

def grdwrite(x, y, z, foutput, zname=None):
    today = datetime.today()
    dataset = Dataset(foutput, 'w', format="NETCDF4")
    dataset.createDimension('x', len(x))
    dataset.createDimension('y', len(y))

    longitud = dataset.createVariable('x', 'f8', 'x')
    latitud = dataset.createVariable('y', 'f8', 'y')

    if zname == None:
        interpolated_values = dataset.createVariable('z', 'f8', ('y', 'x'))
    else:
        interpolated_values = dataset.createVariable(zname, 'f8', ('y', 'x'))

    longitud[:] = x
    latitud[:] = y
    interpolated_values[:, :] = z

    dataset.Conventions = " "
    dataset.title = foutput
    dataset.history = "File written using netCDF4 Python module"
    dataset.description = "Created " + today.strftime("%d/%m/%y")
    dataset.GMT_version = "6.1.0"
    longitud.units = "degrees east"
    latitud.units = "degrees north"
    interpolated_values.units = 'meters'

    dataset.close()
    return


def tif2grd(input_tif, out_grd, nan_value=None):
    with rasterio.open(input_tif) as ds:
        band1 = ds.read(1).astype(float)

        if ds.nodata is not None:
            band1[band1 == ds.nodata] = np.nan
        if nan_value is not None:
            band1 = np.nan_to_num(band1, nan=nan_value)

        height, width = band1.shape
        cols = np.arange(width)
        rows = np.arange(height)

        x, _ = ds.transform * (cols, np.zeros_like(cols))
        _, y = ds.transform * (np.zeros_like(rows), rows)

        y = y[::-1]
        band1 = np.flipud(band1)

        grdwrite(x, y, band1, out_grd)
    return


def clip_tif(input_path, output_path, sw_corner, ne_corner):
    """
    Clips a GeoTIFF file based on South-West and North-East coordinates.
    sw_corner: [longitude, latitude]
    ne_corner: [longitude, latitude]
    """
    # Extract coordinates from the lists
    min_lon, min_lat = sw_corner[0], sw_corner[1]
    max_lon, max_lat = ne_corner[0], ne_corner[1]

    with rasterio.open(input_path) as src:
        window = from_bounds(min_lon, min_lat, max_lon, max_lat, src.transform)

        out_data = src.read(1, window=window)
        out_transform = src.window_transform(window)

        out_meta = src.meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": out_data.shape[0],
            "width": out_data.shape[1],
            "transform": out_transform
        })

        with rasterio.open(output_path, "w", **out_meta) as dest:
            dest.write(out_data, 1)

    print(f"✅ Temporary clipped TIFF created at: {output_path} ")
    return




# --- EXECUTION ---

LOCAL_DIR = 'MessinaWorkshop_THySEA'
input_file = 'gebco_2025_sub_ice_n90.0_s0.0_w0.0_e90.0.tif'
temp_tif = 'temp_clipped.tif'
output_file = 'EM_450m_topobathy.grd'

input_path = os.path.join('/content', LOCAL_DIR, input_file)
temp_path = os.path.join('/content', LOCAL_DIR, temp_tif)
output_path = os.path.join('/content', LOCAL_DIR, output_file)

# New domain
SW = [13.4618467,34.7123333]
NE = [19.1831316,38.9431994]

clip_tif(input_path, temp_path, SW, NE)
tif2grd(temp_path, output_path, nan_value=-9999.0)
print(f"✅ Conversion to grd completed: {output_path}")
os.remove(temp_path)
print(f"✅ Temporary clipped TIFF removed ")

In [ ]:
#Here we define a function to plot .grd grids

from netCDF4 import Dataset
import numpy as np
import matplotlib.pyplot as plt
import os

def plot_bathymetry_grd(file_path):
    """
    Opens a GRD (NetCDF) file using the Dataset module and plots the 'z' variable.
    """

    with Dataset(file_path, mode='r') as rootgrp:
        lons = rootgrp.variables['x'][:]
        lats = rootgrp.variables['y'][:]
        z_data = rootgrp.variables['z'][:]

        # Handle NoData values for plotting (-9999.0 is standard for HySEA)
        z_plot = np.where(z_data == -9999.0, np.nan, z_data)

        plt.figure(figsize=(12, 8))

        cmap = plt.get_cmap('terrain').copy()
        cmap.set_bad(color='white', alpha=0.0)

        im = plt.pcolormesh(lons, lats, z_plot,
                           shading='auto',
                           cmap=cmap)

        plt.colorbar(im, label='Elevation / Depth (m)')

        # Extract filename for the title
        file_name = os.path.basename(file_path)
        plt.title(f"Bathymetry Map (Variable: z)\nFile: {file_name}", fontsize=14)
        plt.xlabel("Longitude [degrees_east]")
        plt.ylabel("Latitude [degrees_north]")
        plt.grid(True, linestyle='--', alpha=0.3)

        plt.tight_layout()
        plt.show()

        # Print grid metadata for verification
        print("-" * 50)
        print(f"File verified: {file_path}")
        print(f"Grid Dimensions (y, x): {z_data.shape}")
        print(f"X Range: {lons.min():.4f} to {lons.max():.4f}")
        print(f"Y Range: {lats.min():.4f} to {lats.max():.4f}")
        print("-" * 50)
    return



# --- Execution Example ---
LOCAL_DIR = 'MessinaWorkshop_THySEA'
output_grd = 'EM_450m_topobathy.grd'
full_path = os.path.join('/content', LOCAL_DIR, output_grd)

plot_bathymetry_grd(full_path)

### **⚠️ Important Note on Data Integrity: Handling NoData Values**

Topobathymetric grids used in any **HySEA** code **must not contain any cells with NoData (NaN) values**. If a single `NaN` exists within the computational domain, the simulation will fail or produce unreasonable results.


The following Python snippet performs a safety check on the grid we have just created. It scans the entire dataset to ensure there are no `NaN` values.

In [ ]:
def check_nodata(finput):
    ds = Dataset(finput)
    elevation = ds.variables['z'][:]
    no_nodata = True
    for fila in elevation:
        nodata = [np.isnan(v) for v in fila]
        if True in nodata:
            no_nodata = False

    if no_nodata == True:
        print("✅ Success: There aren't any NoData (NaN) values in "+finput.split("/")[-1])
    else:
        print("❌ Critical Error: There are NoData (NaN) in "+finput.split("/")[-1])

    ds.close()
    return


LOCAL_DIR = 'MessinaWorkshop_THySEA'

grdfile = "EM_450m_topobathy.grd"
grdfile = os.path.join('/content', LOCAL_DIR, grdfile)
check_nodata(grdfile)

---

# **5. Generate the POIs File**

When performing tsunami propagation simulations, it is common practice to analyze the **time series** of the free surface elevation $\eta$ at specific locations. **Tsunami-HySEA** accepts an input file containing the locations of **Points of Interest (POIs)**—also known as virtual gauges—where high-temporal-resolution data will be extracted.



### **File Specification**
The file format is strict and must follow this structure:

1.  **Header:** The first line must contain a single integer $N$, representing the total number of gauges.
2.  **Body:** The following $N$ lines contain the geographic coordinates ($X$ for Longitude, $Y$ for Latitude) for each point, separated by a space.

The internal structure of the file looks like this:

$$
\begin{matrix}
N \\
X_1 & Y_1 \\
X_2 & Y_2 \\
\dots & \dots \\
X_N & Y_N
\end{matrix}
$$

### **Operational Impact**
* **Temporal Resolution:** While global grid outputs are saved every few minutes (e.g., 600s), POIs can record data much more frequently (e.g., every 15s or 60s), allowing for the capture of the full wave form.
* **Comparison with Data:** These virtual gauges are essential for comparing simulation results with real-world observations from **DART buoys** or coastal **tide gauges**.

> **Note:** Ensure that the coordinates of your POIs fall within the spatial bounds of your topobathymetry grid; otherwise, the model will not be able to extract data for those points.

In [ ]:
# Code to generate the POIs file

import os

LOCAL_DIR = 'MessinaWorkshop_THySEA'
output_file = 'POIs.dat'
output_file = os.path.join('/content', LOCAL_DIR, output_file)

# 1. Define the Coordinates (WGS 84)
# These specific points represents the virtual bouys (lon,lat)
gauges = [
    (16.4907622, 37.5008587),
    (15.7215138, 35.4575427)
]

print(f"📍 Defining {len(gauges)} monitoring stations...")

with open(output_file, 'w') as f:
    f.write(f"{len(gauges)}\n")
    for x, y in gauges:
        f.write(f"{x:.6f} {y:.6f}\n")

print(f"✅ Gauges file saved: {output_file}")

---

# **6. Summary of Simulation Components**

To run a simulation with **Tsunami-HySEA**, you need at least two mandatory components. So far, we have prepared the foundation for our Eastern Mediterranean case study:

1.  **Topobathymetry File (.grd):** * We downloaded global **GEBCO** data in `.tif` format.
    * We clipped the data to our specific area of interest (Eastern Mediterranean).
    * We converted the result into a **NetCDF4 (.grd)** file, ensuring proper variable naming (`x`, `y`, `z`) and removing any `NaN` values.

2.  **Parameter File:** * We examined the basic structure of the configuration file.
    * We established the fixed order of parameters required by the numerical engine.

3.  **Points of Interest (Optional\*):**  Additionally, we created a **POIs** file containing specific coordinates.
---

# **7. Submitting the Simulation to the Queue System**

Once the parameter file is correctly configured with the appropriate paths to the topobathymetry and POIs files, we are almost ready to run the simulation.

### **Final Requirements**
1.  **Load Balancing:** Call the pre-processing executable to optimize the computational domain for GPU execution.
2.  **Shell script to communicate:** Develop a shell script (`.sh`) to communicate with the cluster's workload manager (e.g., Slurm).
---

## **7.1 Executing the Load Balancing Function**

Before running the simulation on the GPU we need to call the **`get_load_balancing`** binary.

This function is **mandatory**. It is a pre-process that divides the domain in optimal sub-domains. Then each sub-domain is assigned to a specific GPU.

### **How it Works**
In the bash command line, we call the `get_load_balancing` executable and pass two main arguments:
1.  **The Parameter File:** (e.g., `parameters.txt`).
2.  **Domain Decomposition:** Number of rows (R) and columns (C) to divide the domain.

`path2binlb/get_load_balancing parameters.txt R C`

### **Single GPU Execution**
For a standard simulation using a **single GPU** we simply divide the domain into 1 row and 1 column:

```bash
# Example for a single GPU setup
path2binlb/get_load_balancing parameters.txt 1 1
```


**⚠️ Important Technical Note**

In the most recent versions of **Tsunami-HySEA**, it is **mandatory to set R = 1**.

In other words, the domain decomposition must always follow the format **(1, n)**, being **n** the number of GPUs to be used. This restriction is due to the recent implementation of **periodic boundary conditions**, which are essential for conducting seamless global tsunami simulations.

---

## **7.2 Creating the Shell Script to Submit the Simulation**

This script serves as the **orchestrator** for the cluster environment. Its primary responsibilities are:

* **Resource Management:** It handles the allocation of computational power (such as **GPU** or CPU cores) and memory on the cluster to ensure the simulation runs at peak efficiency.
* **Execution:** It calls the **TsunamiHySEA** binary, passing the necessary arguments (`parameters.txt`) to trigger the simulation.


> **📌 Note on Environment:** The specific content and format of the present submission script are tailored to the architecture of the **G100 cluster**. If you decide to run this simulation on a different system, you must modify the submission commands and resource headers to match the specific computing environment.

```text
#!/bin/bash

# --- SLURM Directives (Adjusted for G100 cluster) ---
#SBATCH --job-name=THySEA_propagation
#SBATCH --output=simulation.out
#SBATCH --error=simulation.err
#SBATCH --partition=g100_all_serial
#SBATCH --ntasks=1                 # Number of MPI processes
#SBATCH --gres=gpu:1               # Number of GPUs requested (matches ntasks)
#SBATCH --time=00:30:00            # Walltime limit (HH:MM:SS)

# --- Load balancing ---
/g100_work/GeoIn_Messina/SOFTWARE/THySEA_v4.4.0/bin_lb/get_load_balancing parameters.txt 1 1

# --- Execution ---
time mpirun /g100_work/GeoIn_Messina/SOFTWARE/THySEA_v4.4.0/bin/TsunamiHySEA parameters.txt
```

---

## **7.3 How do I run the simulation?**

Log in to the **CINECA G100** cluster using your account.

Load the specific modules required for **Tsunami-HySEA** to interact with the GPU and the NetCDF libraries. Execute the following commands in your terminal:

```bash
module load gcc/10.2.0
module load openmpi/4.1.1--gcc--10.2.0-cuda-11.5.0
module load hdf5/1.10.7--openmpi--4.1.1--gcc--10.2.0
module load parallel-netcdf/1.12.2--openmpi--4.1.1--gcc--10.2.0
module load netcdf-c/4.8.1--openmpi--4.1.1--gcc--10.2.0
```

Create a folder `MyPropagationSimulations` and upload to that folder the files:

`EM_450m_topobathy.grd` --> Topobathymetry file

`parameters.txt`  --> parameters file

`POIs.dat` --> virtual buoys file

`job_EM_propagation450m.sh` --> shell script


Run the command
```bash
sbatch job_EM_propagation450m.sh
```


### **How to check the status:**
Once submitted, you can monitor your simulation's progress using cluster-specific commands (like `squeue -u username` for Slurm) to see if it is **PD** (Pending), **R** (Running), or **CG** (Completing).

---

# **8. Visualizing the Results**

Once the simulation has finished, you will notice that two main output files have been generated in your working directory: `simulation_EM_propagation.nc` and `simulation_EM_propagation_ts.nc`.



### **1. Spatial Data: `simulation_EM_propagation.nc`**
This file contains the variables saved across the entire **2D computational grid**. Based on our parameter file configuration, these include:
* **`eta`:** The free surface elevation (water level) at different time steps.
* **`maximum_eta`:** The maximum wave height reached at each grid point during the entire simulation.
* **`arrival_times`:** The exact time (in seconds) when the tsunami wave first reached each location.

### **2. Time Series Data: `simulation_EM_propagation_ts.nc`**
This is also a NetCDF file, but it is specifically dedicated to the **Points of Interest (POIs)** defined in your `POIs.dat` file.
* It stores the evolution of the water level over time for each virtual buoy.

---

## **8.1. 2D Visualization**

The `simulation_EM_propagation.nc` dataset includes both dynamic and static variables:

### **Time-dependent variables saved**
* **`eta` (Free Surface Elevation):** This is the only variable that has a sequence of frames. By extracting each frame, and putting them sequentially, we can create **animations of the tsunami propagation** near Sicily.



### **Static (non time-dependent) variables saved**
These variables summarize the entire simulation in a single 2D map:
* **`max_height`** This is one of the most critical variables for assessing **hazard levels** and potential impacted zones.
* **`arrival_times`**

---



### **8.1.1 2D Time-Dependent Variables: Animation**

With the following code, we will generate a **GIF animation** showing the evolution of the free surface (`eta`).

### **Temporal Setup**
* **Saving Frequency:** As specified in our `parameters.txt` file (`600.0s`), the model saves a snapshot of the grid every **10 minutes**.
* **Total Duration:** Since we simulated a total of **4 hours** of propagation, the output contains **24 snapshots** of the free surface evolution.

In [ ]:
from netCDF4 import Dataset
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PilImage
import os
import io

import ipywidgets as widgets
from IPython.display import display

# ============================================================
#  CONFIGURATION
# ============================================================
LOCAL_DIR   = 'MessinaWorkshop_THySEA'
nc_path     = os.path.join('/content', LOCAL_DIR, 'simulation_EM_propagation.nc')
VMIN, VMAX  = -0.25, 0.25
FPS         = 1
WIDTH       = '60%'
# ============================================================


def load_and_prerender(file_path, vmin=None, vmax=None):
    rootgrp    = Dataset(file_path, mode='r')
    lons       = rootgrp.variables['lon'][:]
    lats       = rootgrp.variables['lat'][:]
    times      = rootgrp.variables['time'][:]
    eta_var    = rootgrp.variables['eta']
    num_frames = len(times)

    if vmin is None or vmax is None:
        print("EEstimating color scale...")
        s1   = np.where(eta_var[0, :, :]           == -9999.0, np.nan, eta_var[0, :, :])
        s2   = np.where(eta_var[num_frames//2,:,:] == -9999.0, np.nan, eta_var[num_frames//2,:,:])
        vmax = float(np.nanmax([np.nanmax(s1), np.nanmax(s2)]))
        vmin = -vmax

    print(f"pre-rendering {num_frames} frames...")
    plt.ioff()
    fig, ax = plt.subplots(figsize=(11, 6))
    mesh = ax.pcolormesh(lons, lats, eta_var[0, :, :],
                         shading='auto', cmap='RdBu_r',
                         vmin=vmin, vmax=vmax)
    plt.colorbar(mesh, ax=ax, label='wave amplitude (m)')
    title_obj = ax.set_title("")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.tight_layout()

    png_frames = []
    pil_frames = []

    for i in range(num_frames):
        fd = np.array(eta_var[i, :, :])
        fd = np.where(fd == -9999.0, np.nan, fd)
        mesh.set_array(fd.ravel())
        title_obj.set_text(f"Tsunami propagation | Time: {times[i]:.1f} s")

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        raw = buf.read()
        png_frames.append(raw)

        # Reutilizar el mismo buffer para PIL
        pil_frames.append(PilImage.open(io.BytesIO(raw)).convert('RGB'))

        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{num_frames} frames ready")

    plt.close(fig)
    rootgrp.close()
    print("✅ pre-rendered completed.")
    return lons, lats, times, png_frames, pil_frames, vmin, vmax


def build_gif(pil_frames, fps):
    print("Generating GIF...")
    buf = io.BytesIO()
    pil_frames[0].save(
        buf,
        format='GIF',
        save_all=True,
        append_images=pil_frames[1:],
        duration=int(1000 / fps),
        loop=0,
        optimize=False,
    )
    buf.seek(0)
    print("✅ GIF ready.")
    return buf.read()





def create_viewer(file_path, vmin=None, vmax=None, fps=FPS, width=WIDTH):

    lons, lats, times, png_frames, pil_frames, v_min, v_max = \
        load_and_prerender(file_path, vmin, vmax)
    num_frames = len(png_frames)
    gif_bytes  = build_gif(pil_frames, fps)

    # ------------------------------------------------------------------
    # Sección 1: Interactive viewer with slider
    # ------------------------------------------------------------------
    img_widget = widgets.Image(
        value=png_frames[0],
        format='png',
        layout=widgets.Layout(width=width)
    )

    time_label = widgets.Label(
        value=f"⏱  t = {times[0]:.1f} s  |  Frame 1 / {num_frames}"
    )

    slider = widgets.IntSlider(
        value=0, min=0, max=num_frames - 1, step=1,
        description='Frame:',
        continuous_update=True,
        layout=widgets.Layout(width=width)
    )

    btn_rewind  = widgets.Button(description='⏮ Start', button_style='info', layout=widgets.Layout(width='110px'))
    btn_back    = widgets.Button(description='⏪ -1',     button_style='',     layout=widgets.Layout(width='90px'))
    btn_forward = widgets.Button(description='+1 ⏩',     button_style='',     layout=widgets.Layout(width='90px'))
    btn_end     = widgets.Button(description='End ⏭',  button_style='info', layout=widgets.Layout(width='110px'))

    controls = widgets.HBox(
        [btn_rewind, btn_back, btn_forward, btn_end],
        layout=widgets.Layout(justify_content='center', margin='4px 0')
    )

    section_slider = widgets.VBox(
        [
            widgets.HTML('<h3 style="text-align:center">🖱️ Visor interactivo</h3>'),
            img_widget,
            time_label,
            slider,
            controls,
        ],
        layout=widgets.Layout(align_items='center', width='100%')
    )

    def on_slider_change(change):
        idx = int(change['new'])
        img_widget.value = png_frames[idx]
        time_label.value = f"⏱  t = {times[idx]:.1f} s  |  Frame {idx+1} / {num_frames}"

    def on_rewind(b):  slider.value = 0
    def on_back(b):    slider.value = max(0, slider.value - 1)
    def on_forward(b): slider.value = min(num_frames - 1, slider.value + 1)
    def on_end(b):     slider.value = num_frames - 1

    slider.observe(on_slider_change, names='value')
    btn_rewind.on_click(on_rewind)
    btn_back.on_click(on_back)
    btn_forward.on_click(on_forward)
    btn_end.on_click(on_end)

    # ------------------------------------------------------------------
    # Sección 2: Animación GIF
    # ------------------------------------------------------------------
    gif_widget = widgets.Image(
        value=gif_bytes,
        format='gif',
        layout=widgets.Layout(width=width)
    )

    section_gif = widgets.VBox(
        [
            widgets.HTML(f'<h3 style="text-align:center">▶️ Animación ({fps} FPS)</h3>'),
            gif_widget,
        ],
        layout=widgets.Layout(align_items='center', width='100%', margin_top='20px')
    )

    # ------------------------------------------------------------------
    # Layout final
    # ------------------------------------------------------------------
    divider = widgets.HTML('<hr style="width:80%;border-top:2px solid #ccc;margin:20px auto"/>')

    display(widgets.VBox([section_slider, divider, section_gif]))


# --- Ejecución ---
create_viewer(nc_path, vmin=VMIN, vmax=VMAX, fps=FPS, width=WIDTH)

---
## 8.1.2 Static variables

With the following code we will generate an image showing the 2D distribution of the maximum water height and the arrival times of the water waves.

In [ ]:
from netCDF4 import Dataset
import numpy as np
import matplotlib.pyplot as plt
import os

# --- Load the Dataset ---
LOCAL_DIR = 'MessinaWorkshop_THySEA'
nc_path = os.path.join('/content', LOCAL_DIR, 'simulation_EM_propagation.nc')

def plot_max_height(file_path,vmax=None):
    """
    Plots a static map of maximum wave amplitude with updated Matplotlib syntax.
    Handles NoData as transparent and uses a symmetric RdBu_r colormap.
    """
    with Dataset(file_path, mode='r') as rootgrp:
        lons = rootgrp.variables['lon'][:]
        lats = rootgrp.variables['lat'][:]
        max_h = rootgrp.variables['max_height'][:]


        # --- Updated Colormap Syntax ---
        cmap = plt.get_cmap('RdBu_r').copy()

        # Set transparent color for NaN values (land/no-data)
        cmap.set_bad(color='black', alpha=1.0)

        v_max = np.nanmax(np.abs(max_h))
        if vmax is None:
          v_limit = v_max
        else:
          v_limit = min(v_max, vmax)

        # --- Plotting ---
        plt.figure(figsize=(14, 10))

        # pcolormesh is efficient for large HySEA grids
        im = plt.pcolormesh(lons, lats, max_h,
                          shading='auto',
                          cmap=cmap,
                          vmin=0,
                          vmax=v_limit)


        plt.colorbar(im, label='Maximum Amplitude (m)')
        plt.title("Tsunami Maximum Wave Amplitude", fontsize=16)
        plt.xlabel("Longitude [degrees_east]")
        plt.ylabel("Latitude [degrees_north]")
        plt.grid(True, linestyle='--', alpha=0.3)

        plt.tight_layout()
        plt.show()
    return


#plot_max_height(nc_path)
plot_max_height(nc_path,vmax=3)




In [ ]:
from netCDF4 import Dataset
import numpy as np
import matplotlib.pyplot as plt
import os


def plot_arrival_times_adaptive(file_path,maxtime_min,step_min):
    """
    Plots arrival times with a Red-to-Blue gradient, adapted to the actual data range.
    Automatically determines contour levels and time steps.
    """
    with Dataset(file_path, mode='r') as rootgrp:
        lons = rootgrp.variables['lon'][:]
        lats = rootgrp.variables['lat'][:]
        arrival_sec = rootgrp.variables['arrival_times'][:]

        # Pre-processing: Convert fill value to NaN and seconds to minutes
        arrival_sec = np.where(arrival_sec == -9999.0, np.nan, arrival_sec)
        arrival_min = arrival_sec / 60.0

        # --- Dynamic Range Calculation ---
        # Get the real max time, ignoring NaNs
        max_time = np.nanmax(arrival_min)

        # Decide step size based on duration:
        # If less than 2h, steps of 10 min. If more, steps of 15 or 30 min.
        if max_time <= 120:
            step = 5
        elif max_time <= 300:
            step = 15
        else:
            step = 30

        # Create levels from 0 to max_time (rounded up to the next step)
        levels = np.arange(0, maxtime_min, step_min)

        # --- Plotting Setup ---
        plt.figure(figsize=(14, 10))

        # 'RdYlBu' provides Red (low/fast) -> Yellow -> Blue (high/slow)
        cmap = plt.get_cmap('RdYlBu').copy()
        cmap.set_bad(color='white', alpha=0.0) # Transparent land

        masked_arrival = np.ma.masked_invalid(arrival_min)

        # 1. Background filled contours
        cfm = plt.contourf(lons, lats, masked_arrival, levels=levels, cmap=cmap, extend='max')

        # 2. Isochrone Lines
        cl = plt.contour(lons, lats, masked_arrival, levels=levels, colors='black', linewidths=0.6, alpha=0.4)

        # Add labels directly to the lines
        # Isochrone Lines
        cl = plt.contour(lons, lats, masked_arrival, levels=levels, colors='black', linewidths=0.6, alpha=0.4)

        # Generamos etiquetas y las guardamos en una variable
        labels = plt.clabel(cl, inline=True, fontsize=10, fmt='%1.0f min', colors='black')

        # Aplicamos la negrita manualmente a cada etiqueta
        for txt in labels:
            txt.set_fontweight('bold')

        # --- Formatting ---
        # Adaptive ticks: if too many levels, show every second tick
        display_ticks = levels if len(levels) < 15 else levels[::2]
        cbar = plt.colorbar(cfm, ticks=display_ticks)
        cbar.set_label('Arrival Time (minutes)')

        plt.title(f"Tsunami Arrival Times (Max: {max_time:.1f} min)", fontsize=16)
        plt.xlabel("Longitude [degrees_east]")
        plt.ylabel("Latitude [degrees_north]")
        plt.grid(True, linestyle=':', alpha=0.3)

        plt.tight_layout()
        plt.show()

# --- Execution ---
LOCAL_DIR = 'MessinaWorkshop_THySEA'
nc_path = os.path.join('/content', LOCAL_DIR, 'simulation_EM_propagation.nc')

if os.path.exists(nc_path):
    plot_arrival_times_adaptive(nc_path,40,1)

---

## **8.2. Time Series Visualization**

Since we provided the `POIs.dat` file in our `parameters.txt` configuration, **Tsunami-HySEA** has produced an additional output file named `simulation_EM_propagation_ts.nc`.

This file contains the evolution of the free surface $\eta$ specifically at the coordinates of each virtual buoy.



The following Python code performs two main tasks to help analyze the gauge data:

1.  It generates a spatial map showing the exact **geographic locations** of the virtual buoys (POIs) that were included in our configuration.
2.  It creates a folder named `TS_Plots/` and automatically generates an individual **time series plot** for every single point of interest.

In [ ]:
from netCDF4 import Dataset
import numpy as np
import matplotlib.pyplot as plt
import os
import shutil

# --- Configuration ---
LOCAL_DIR = 'MessinaWorkshop_THySEA'
bathy_file = os.path.join('/content', LOCAL_DIR, 'EM_450m_topobathy.grd')
nc_pois_path = os.path.join('/content', LOCAL_DIR, 'simulation_EM_propagation_ts.nc')
output_folder = os.path.join('/content', LOCAL_DIR, 'TS_Plots')

# Create the output directory (clean it if it exists)
if os.path.exists(output_folder):
    shutil.rmtree(output_folder)
os.makedirs(output_folder)

# --- 1. Plot Map with Buoy Locations (Display on Screen) ---
def plot_buoy_map(bathy_path, pois_path):
    with Dataset(bathy_path, 'r') as b_ds, Dataset(pois_path, 'r') as p_ds:
        # Bathymetry data (assuming standard NetCDF structure for .grd)
        lon_b = b_ds.variables['x'][:]
        lat_b = b_ds.variables['y'][:]
        # In some .grd files the variable is 'z' or 'elevation'
        z_name = 'z' if 'z' in b_ds.variables else list(b_ds.data_vars)[0]
        z = b_ds.variables[z_name][:]

        # Buoy locations
        lon_p = p_ds.variables['longitude'][:]
        lat_p = p_ds.variables['latitude'][:]

        plt.figure(figsize=(12, 8))
        # Plot terrain
        plt.pcolormesh(lon_b, lat_b, z, cmap='terrain', shading='auto', vmin=-5000, vmax=2000)
        plt.colorbar(label='Elevation (m)')

        # Plot Buoys
        plt.scatter(lon_p, lat_p, color='red', edgecolor='white', s=50, zorder=5)

        # Add labels for each buoy
        for i, (ln, lt) in enumerate(zip(lon_p, lat_p)):
            plt.text(ln + 0.1, lt + 0.1, f"{i+1}", color='white',
                     fontsize=12, fontweight='bold',
                     bbox=dict(facecolor='black', alpha=0.5, edgecolor='none'))

        plt.title("Virtual Buoys Locations Map", fontsize=14)
        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.show()
    return

# --- 2. Save Time Series as PNGs (Silent Execution) ---
def save_buoy_series(pois_path, out_dir):
    with Dataset(pois_path, 'r') as ds:
        times = ds.variables['time'][:] / 60.0  # Convert to minutes
        eta = ds.variables['eta'][:]
        lons = ds.variables['longitude'][:]
        lats = ds.variables['latitude'][:]

        num_buoys = eta.shape[1]
        print(f"Generating {num_buoys} time series plots in {out_dir}...")

        for i in range(num_buoys):
            buoy_eta = np.where(eta[:, i] == -9999.0, np.nan, eta[:, i])

            fig, ax = plt.subplots(figsize=(10, 4))
            ax.plot(times, buoy_eta, color='blue', linewidth=1)
            ax.axhline(0, color='black', lw=0.5, ls='--')

            ax.set_title(f"Buoy {i+1} | Lon: {lons[i]:.3f} Lat: {lats[i]:.3f}")
            ax.set_xlabel("Time (min)")
            ax.set_ylabel("Eta (m)")
            ax.grid(True, alpha=0.3)

            # Save and close immediately to save RAM and not show in notebook
            filename = os.path.join(out_dir, f"buoy_{i+1}_timeseries.png")
            plt.savefig(filename, dpi=150, bbox_inches='tight')
            plt.close(fig)

    print("✅ All PNGs saved successfully.")


plot_buoy_map(bathy_file, nc_pois_path)
save_buoy_series(nc_pois_path, output_folder)

---

# **9. Nested Grid Simulations**

This section demonstrates how to construct and implement **nested grids** within a single simulation to analyze high-resolution tsunami impacts and coastal inundation.

*A tsunami simulation is only as good as the topobathymetric data it uses.*

### **The Importance of Grid Preparation**
The most labor-intensive part of conducting a high-resolution tsunami simulation—especially one focused on coastal inundation—is the preparation of the input data. Specifically, the creation of accurate, well-aligned grids is critical for numerical stability.


### **Where to find bathymetric data?**
In general, acquiring **high-resolution bathymetric data** (5m, 10m, 20m) is challenging. These datasets are usually associated with privately funded bathymetric surveys and typically cover only small areas.


### **Where to find topographic data?**
On the other hand, obtaining high-resolution topographic data is often easier, depending on the country. In **Spain**, we have access to **Digital Terrain Models (DTM)** with resolutions up to 1m based on LiDAR data provided by the IGN, freely available at the **CNIG** [Download Center](https://centrodedescargas.cnig.es/CentroDescargas/modelo-digital-terreno-mdt01).

---

## **9.1 Retrieving High-Resolution Bathymetry**

For European waters, **EMODnet** provides high-quality bathymetry data with a resolution of **1/16 arc-minutes** (approximately **112.5m**). This data can be accessed and downloaded via the [EMODnet Bathymetry Geoviewer](https://emodnet.ec.europa.eu/geoviewer/).

### **The Challenge of Coastal Data Integration**
While EMODnet is an excellent resource, it primarily provides **bathymetric** (underwater) data. To perform accurate **inundation simulations**, we must merge this with a **topographic** (land) dataset. This merging process is complex and requires:

* **Precise Coastline Definition:** A high-quality shoreline vector to act as the "seam" between datasets.
* **Resolution Alignment:** Ensuring that both the topography and bathymetry datasets have comparable spatial resolutions to avoid interpolation artifacts.
* **Specialized Processing:** Utilizing Geographic Information Systems (GIS) software (e.g., ArcGIS, QGIS) or spatial Python libraries to merge the rasters into a single, seamless topobathymetric product.



### **Current Objective**
In this exercise, we will simplify the process by creating a nested grid in an area **entirely covered by water**. This ensures that our nested grid contains no `NoData` values, allowing us to focus on the multi-level numerical coupling.

**Workflow:**
1.  Download the EMODnet `.tif` file, covering a similar area as our level 0 grid `EM_450m_topobathy.grd`.
2.  Clip the data to our coastal area of interest.
3.  Convert the clipped raster to **.grd (NetCDF4)** format for Tsunami-HySEA compatibility.

In [ ]:
import rasterio
from netCDF4 import Dataset
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import os


#--------Convert the .tif file to .grd---------
LOCAL_DIR = 'MessinaWorkshop_THySEA'
input_file = 'F6_2024.tif'
temp_file = 'F6_clipped.tif'
output_file = 'F6_clipped.grd'

input_path = os.path.join('/content', LOCAL_DIR, input_file)
temp_path = os.path.join('/content', LOCAL_DIR, temp_file)
output_path = os.path.join('/content', LOCAL_DIR, output_file)

SW = [15.7935924,35.9294381]
NE = [16.7683840,37.8002502]
clip_tif(input_path, temp_path, SW, NE)

tif2grd(temp_path, output_path)
os.remove(temp_path)

plot_bathymetry_grd(output_path)
check_nodata(output_path)

## **9.2 Creating the Nested Grids**

In this step, we use a specialized manually-coded Python function called `nest_down`. This tool allows us to generate a high-resolution sub-grid that is perfectly embedded within our primary domain (`EM_450m_topobathy.grd`).

### **How `nest_down` Works**
To ensure the simulation runs correctly, the nested grid must satisfy specific geometric requirements:
* **Spatial Alignment:** The boundaries of the new high-resolution grid must align precisely with the cell edges of the coarser "parent" grid.
* **Resolution Ratio:** The resolution increase must follow a fixed power of 2 ratio (e.g., 2, 4, 8, ...), ensuring that one coarse cell corresponds exactly to a set number of fine cells.
* **Interpolation:** The function uses **Regular Grid Interpolation** to map the high-resolution EMODnet data onto the new nested structure while maintaining consistency with the surrounding coarser bathymetry.


In [ ]:
import itertools
import time
import os

from netCDF4 import Dataset
from matplotlib import path

from scipy.spatial import ConvexHull
from scipy.interpolate import RegularGridInterpolator


def grdread(grdfile):
    """
    ENTRADA: fichero .grd
    SALIDA: vectores x,y junto con la matriz z
    """
    ds = Dataset(grdfile)
    var_names = list(ds.variables.keys())

    ejex = ["lon", "x", "longitude"]
    ejey = ["lat", "y", "latitude"]
    ejez = ["topo", "z", "Band1"]

    for namex in var_names:
        if namex in ejex:
            x = ds[namex][:]
    for namey in var_names:
        if namey in ejey:
            y = ds[namey][:]
    for namez in var_names:
        if namez in ejez:
            z = ds[namez][:]

    dx = x[1]-x[0]
    dy = y[1]-y[0]

    return [[x,y,z],dx,dy]






def nest_down(finemeshfile, coarsemeshfile, ratio, SW, NE, foutput):

    # Creamos la malla grosera
    class G:
        pass

    G.coarsemeshfile = coarsemeshfile
    G.x = []
    G.y = []
    G.Z = []

    G.x, G.y, G.Z = grdread(G.coarsemeshfile)[0]
    dx_grosera = grdread(G.coarsemeshfile)[1]

    G.X, G.Y = np.meshgrid(G.x, G.y)


    # Creamos la malla fina
    class F:
        pass

    F.finemeshfile = finemeshfile
    F.x = []
    F.y = []
    F.Z = []

    F.x, F.y, F.Z = grdread(F.finemeshfile)[0]
    dx_fina = grdread(F.finemeshfile)[1]
    dy_fina = grdread(F.finemeshfile)[2]

    F.X, F.Y = np.meshgrid(F.x, F.y)

    # Definimos la envolvente convexa de la malla fina (sera necesaria a continuacion)
    u1 = np.ma.compress_cols(F.X)
    v1 = np.ma.compress_cols(F.Y)
    u1 = u1.flatten()
    v1 = v1.flatten()
    points_fina2D_modificado = np.vstack([u1,v1]).T
    hull_fina_modificado = ConvexHull(points_fina2D_modificado)
    k = hull_fina_modificado.vertices

    points_fina2D_modificado[k[0]][0] = points_fina2D_modificado[k[0]][0]-dx_fina/2
    points_fina2D_modificado[k[0]][1] = points_fina2D_modificado[k[0]][1]-dy_fina/2

    points_fina2D_modificado[k[1]][0] = points_fina2D_modificado[k[1]][0]+dx_fina/2
    points_fina2D_modificado[k[1]][1] = points_fina2D_modificado[k[1]][1]-dy_fina/2

    points_fina2D_modificado[k[2]][0] = points_fina2D_modificado[k[2]][0]+dx_fina/2
    points_fina2D_modificado[k[2]][1] = points_fina2D_modificado[k[2]][1]+dy_fina/2

    points_fina2D_modificado[k[3]][0] = points_fina2D_modificado[k[3]][0]-dx_fina/2
    points_fina2D_modificado[k[3]][1] = points_fina2D_modificado[k[3]][1]+dy_fina/2

    LLB_fina = points_fina2D_modificado[k[0]]
    LRB_fina = points_fina2D_modificado[k[1]]
    URB_fina = points_fina2D_modificado[k[2]]
    ULB_fina = points_fina2D_modificado[k[3]]


    if SW[0]<LLB_fina[0] or SW[1]<LLB_fina[1] or NE[0]>URB_fina[0] or NE[1]>URB_fina[1]:
        print("The rectangle defined by the user is not fully contained within the fine mesh")
        print("Be aware that NoData values may appear in the resulting mesh")


    # Definimos los extremos del rectangulo dado por el usuario SW/SE/NE/NW
    SW = np.array(SW)
    NE = np.array(NE)
    SE = np.array([NE[0], SW[1]])
    NW = np.array([SW[0], NE[1]])


    xxc = G.x
    yyc = G.y
    pts_malla_gruesa = np.array(list(itertools.product(xxc, yyc)))



    # Comprobamos qué puntos de la grosera estan dentro de las coordenadas dadas por el usuario
    p = path.Path([SW, SE, NE, NW])
    inG = p.contains_points(pts_malla_gruesa)

    G.X = G.X.T
    G.X = G.X.flatten()
    xnew = G.X[inG]     # Puntos x de la grosera dentro de las coordenadas

    G.Y = G.Y.T
    G.Y = G.Y.flatten()
    ynew = G.Y[inG]     # Puntos y de la grosera dentro de las coordenadas

    G.Z = G.Z.T
    G.Z = G.Z.flatten()
    znew = G.Z[inG]     # Puntos z de la grosera dentro de las coordenadas


    Ax = np.unique(xnew)
    Ay = np.unique(ynew)
    dx = dx_grosera
    dy = Ay[1]-Ay[0]

    if (Ax[0]-dx/2.)<LLB_fina[0]:
        Ax = Ax[1:]
    if (Ax[-1]+dx/2.)>URB_fina[0]:
        Ax = Ax[0:-1]
    if (Ay[0]-dy/2.)<LLB_fina[1]:
        Ay = Ay[1:]
    if (Ay[-1]+dy/2.)>URB_fina[1]:
        Ay = Ay[0:-1]

    xx_new = np.arange(Ax[0] - dx/2 + dx/(2*ratio), Ax[-1] + dx/2 - dx/(4*ratio), dx/ratio)
    yy_new = np.arange(Ay[0] - dy/2 + dy/(2*ratio), Ay[-1] + dy/2 - dy/(4*ratio), dy/ratio)


    valores = F.Z.T
    print("Interpolating values... (it might take some minutes)")

    interp = RegularGridInterpolator((F.x, F.y), valores)
    new_points = np.array(list(itertools.product(xx_new,yy_new)))

    Nrow = len(yy_new)
    Ncolumn = len(xx_new)
    Zc=interp(new_points)
    Zc=Zc.reshape(Ncolumn, Nrow).T

    grdwrite(xx_new, yy_new, Zc, foutput)

    return foutput


LOCAL_DIR = 'MessinaWorkshop_THySEA'
coarsemesh_file = 'EM_450m_topobathy.grd'
finemesh_file = 'F6_clipped.grd'
output_file = 'EM_112m_bathy.grd'

coarsemesh_path = os.path.join('/content', LOCAL_DIR, coarsemesh_file)
finemesh_path = os.path.join('/content', LOCAL_DIR, finemesh_file)
output_path = os.path.join('/content', LOCAL_DIR, output_file)

SW = [15.85,36.0]
NE = [16.65,37.75]

ratio = 4
start=time.time()
nest_down(finemesh_path, coarsemesh_path, ratio, SW, NE, output_path)
end = time.time()
print(end-start)
check_nodata(output_path)

## **9.3 Configuring the New Parameter File**

To execute a multi-level simulation, the parameter file must be updated to inform the numerical engine about the nested grid hierarchy. This file acts as the roadmap for the simulation, defining how the grids interact and what data is recorded at each level.


```text
MEDITERRANEAN_2LEVELS          # name of the experiment
EM_450m_topobathy.grd          # bathymetry file
1                              # initialization mode 1: Okada Standard
0                              # apply kajura filter?
1                              # number of faults
0.0 14.9887 37.1526 9.802863566673047 85.84465449734515 22.98264453386507 67.5 50.0 90.0 3.9955153462891992  # okada parameters (time, lon, lat, depth, length, width, strike, dip, rake, slip)
0                              # crop deformation?
simulation_EM_propagation_L0   # output filename for Level 0
0 0 0 0 0 0 0 0 0 0            # output variables L0
2                              # number of levels
4                              # refinement ratio
1                              # number of submeshes
EM_112m_bathy.grd              # nested grid file (Level 1)
simulation_EM_propagation_L1   # output filename for Level 1
1 1 0 0 0 0 0 0 0 1            # output variables L1 (eta, max_eta, arrival)
1                              # upper boundary condition (1:open, -1:wall)
1                              # lower boundary condition (1:open, -1:wall)
1                              # left boundary condition (1:open, -1:wall)
1                              # right boundary condition (1:open, -1:wall)
14400.1                        # total simulation time (seconds)
600.0                          # netcdf saving time step (seconds)
1                              # read points from file? (for timeseries)
POIs.dat                       # file with the location of Points Of Interest
60.0                           # timeseries saving time step (seconds)
0.5                            # CFL
5e-3                           # epsilon_h
20.0                           # threshold for WAF scheme (meters)
0.2                            # stability coefficient
0                              # friction type
0.02                           # Manning bottom friction
100.0                          # maximum allowed velocity of water
1000000.0                      # typical length
1000.0                         # typical height
1e-3                           # threshold for arrival times

```

## **9.4 Creating the new Shell Script**

To execute a nested grid simulation on the cluster, we must update our job submission script with the new parameters file. This script acts as the bridge between our configuration files and the **High-Performance Computing (HPC)** hardware.

```text
#!/bin/bash

# --- SLURM Directives (Adjusted for G100 cluster) ---
#SBATCH --job-name=THySEA_propagation2Levels
#SBATCH --output=simulation2Levels.out
#SBATCH --error=simulation2Levels.err
#SBATCH --partition=g100_all_serial
#SBATCH --ntasks=1                 # Number of MPI processes
#SBATCH --gres=gpu:1               # Number of GPUs requested (matches ntasks)
#SBATCH --time=00:30:00            # Walltime limit (HH:MM:SS)

# --- Load balancing ---
/g100_work/GeoIn_Messina/SOFTWARE/THySEA_v4.4.0/bin_lb/get_load_balancing parametersNested.txt 1 1

# --- Execution ---
time mpirun /g100_work/GeoIn_Messina/SOFTWARE/THySEA_v4.4.0/bin/TsunamiHySEA parametersNested.txt
```

---

## **9.5 How do I run the simulation?**

Log in to the **CINECA G100** cluster using your account.

Load the specific modules required for **Tsunami-HySEA** to interact with the GPU and the NetCDF libraries. Execute the following commands in your terminal:

```bash
module load gcc/10.2.0
module load openmpi/4.1.1--gcc--10.2.0-cuda-11.5.0
module load hdf5/1.10.7--openmpi--4.1.1--gcc--10.2.0
module load parallel-netcdf/1.12.2--openmpi--4.1.1--gcc--10.2.0
module load netcdf-c/4.8.1--openmpi--4.1.1--gcc--10.2.0
```

Create a folder `MyNestedSimulations` and upload to that folder the files:

`EM_450m_topobathy.grd` --> Level 0 grid

`EM_112m_bathy.grd`     --> Level 1 grid (nested grid, higher-res)

`parametersNested.txt`  --> parameters file

`POIs.dat` --> virtual buoys file

`job_EM_propagation112m.sh` --> shell script  


Run the command
```bash
sbatch job_EM_propagation112m.sh
```

---

## **9.6 Visualizing the Nested Results**

Once the nested grid simulation is complete, we can analyze the high-resolution outputs generated for the nested domain. We can utilize the visualization functions defined in previous sections to inspect the Level 1 grid data stored in `simulation_EM_propagation_L1.nc`.

In [ ]:
from netCDF4 import Dataset
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PilImage
import os
import io

import ipywidgets as widgets
from IPython.display import display

# ============================================================
#  CONFIGURATION
# ============================================================
LOCAL_DIR   = 'MessinaWorkshop_THySEA'
nc_path     = os.path.join('/content', LOCAL_DIR, 'simulation_EM_propagation_L1.nc')
VMIN, VMAX  = -0.25, 0.25
FPS         = 1
WIDTH       = '60%'
# ============================================================


def load_and_prerender(file_path, vmin=None, vmax=None):
    rootgrp    = Dataset(file_path, mode='r')
    lons       = rootgrp.variables['lon'][:]
    lats       = rootgrp.variables['lat'][:]
    times      = rootgrp.variables['time'][:]
    eta_var    = rootgrp.variables['eta']
    num_frames = len(times)

    if vmin is None or vmax is None:
        print("EEstimating color scale...")
        s1   = np.where(eta_var[0, :, :]           == -9999.0, np.nan, eta_var[0, :, :])
        s2   = np.where(eta_var[num_frames//2,:,:] == -9999.0, np.nan, eta_var[num_frames//2,:,:])
        vmax = float(np.nanmax([np.nanmax(s1), np.nanmax(s2)]))
        vmin = -vmax

    print(f"pre-rendering {num_frames} frames...")
    plt.ioff()
    fig, ax = plt.subplots(figsize=(11, 6))
    mesh = ax.pcolormesh(lons, lats, eta_var[0, :, :],
                         shading='auto', cmap='RdBu_r',
                         vmin=vmin, vmax=vmax)
    plt.colorbar(mesh, ax=ax, label='wave amplitude (m)')
    title_obj = ax.set_title("")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.tight_layout()

    png_frames = []
    pil_frames = []

    for i in range(num_frames):
        fd = np.array(eta_var[i, :, :])
        fd = np.where(fd == -9999.0, np.nan, fd)
        mesh.set_array(fd.ravel())
        title_obj.set_text(f"Tsunami propagation | Time: {times[i]:.1f} s")

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        raw = buf.read()
        png_frames.append(raw)

        # Reutilizar el mismo buffer para PIL
        pil_frames.append(PilImage.open(io.BytesIO(raw)).convert('RGB'))

        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{num_frames} frames ready")

    plt.close(fig)
    rootgrp.close()
    print("✅ pre-rendered completed.")
    return lons, lats, times, png_frames, pil_frames, vmin, vmax


def build_gif(pil_frames, fps):
    print("Generating GIF...")
    buf = io.BytesIO()
    pil_frames[0].save(
        buf,
        format='GIF',
        save_all=True,
        append_images=pil_frames[1:],
        duration=int(1000 / fps),
        loop=0,
        optimize=False,
    )
    buf.seek(0)
    print("✅ GIF ready.")
    return buf.read()





def create_viewer(file_path, vmin=None, vmax=None, fps=FPS, width=WIDTH):

    lons, lats, times, png_frames, pil_frames, v_min, v_max = \
        load_and_prerender(file_path, vmin, vmax)
    num_frames = len(png_frames)
    gif_bytes  = build_gif(pil_frames, fps)

    # ------------------------------------------------------------------
    # Sección 1: Interactive viewer with slider
    # ------------------------------------------------------------------
    img_widget = widgets.Image(
        value=png_frames[0],
        format='png',
        layout=widgets.Layout(width=width)
    )

    time_label = widgets.Label(
        value=f"⏱  t = {times[0]:.1f} s  |  Frame 1 / {num_frames}"
    )

    slider = widgets.IntSlider(
        value=0, min=0, max=num_frames - 1, step=1,
        description='Frame:',
        continuous_update=True,
        layout=widgets.Layout(width=width)
    )

    btn_rewind  = widgets.Button(description='⏮ Start', button_style='info', layout=widgets.Layout(width='110px'))
    btn_back    = widgets.Button(description='⏪ -1',     button_style='',     layout=widgets.Layout(width='90px'))
    btn_forward = widgets.Button(description='+1 ⏩',     button_style='',     layout=widgets.Layout(width='90px'))
    btn_end     = widgets.Button(description='End ⏭',  button_style='info', layout=widgets.Layout(width='110px'))

    controls = widgets.HBox(
        [btn_rewind, btn_back, btn_forward, btn_end],
        layout=widgets.Layout(justify_content='center', margin='4px 0')
    )

    section_slider = widgets.VBox(
        [
            widgets.HTML('<h3 style="text-align:center">🖱️ Visor interactivo</h3>'),
            img_widget,
            time_label,
            slider,
            controls,
        ],
        layout=widgets.Layout(align_items='center', width='100%')
    )

    def on_slider_change(change):
        idx = int(change['new'])
        img_widget.value = png_frames[idx]
        time_label.value = f"⏱  t = {times[idx]:.1f} s  |  Frame {idx+1} / {num_frames}"

    def on_rewind(b):  slider.value = 0
    def on_back(b):    slider.value = max(0, slider.value - 1)
    def on_forward(b): slider.value = min(num_frames - 1, slider.value + 1)
    def on_end(b):     slider.value = num_frames - 1

    slider.observe(on_slider_change, names='value')
    btn_rewind.on_click(on_rewind)
    btn_back.on_click(on_back)
    btn_forward.on_click(on_forward)
    btn_end.on_click(on_end)

    # ------------------------------------------------------------------
    # Sección 2: Animación GIF
    # ------------------------------------------------------------------
    gif_widget = widgets.Image(
        value=gif_bytes,
        format='gif',
        layout=widgets.Layout(width=width)
    )

    section_gif = widgets.VBox(
        [
            widgets.HTML(f'<h3 style="text-align:center">▶️ Animación ({fps} FPS)</h3>'),
            gif_widget,
        ],
        layout=widgets.Layout(align_items='center', width='100%', margin_top='20px')
    )

    # ------------------------------------------------------------------
    # Layout final
    # ------------------------------------------------------------------
    divider = widgets.HTML('<hr style="width:80%;border-top:2px solid #ccc;margin:20px auto"/>')

    display(widgets.VBox([section_slider, divider, section_gif]))


# --- Ejecución ---
create_viewer(nc_path, vmin=VMIN, vmax=VMAX, fps=FPS, width=WIDTH)